# Модель SIR: Исследование эффекта миграции

**Цель работы:** Исследовать, как интенсивность перемещения людей между городами
влияет на скорость распространения эпидемии (время достижения пика) и масштаб пика.

Инфекция начинается только в одном городе, остальные изначально здоровы.

## 1. Инициализация проекта и загрузка пакетов

In [ ]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots, CSV, Random, Statistics
include(srcdir("sir_model.jl"))

## 2. Функция создания матрицы миграции

Создаёт матрицу вероятностей миграции между городами.
- `C` — количество городов
- `intensity` — вероятность переехать в другой город (оставшаяся вероятность — остаться)

При интенсивности 0 все остаются на месте.
При интенсивности 1 вероятность равномерно распределена между всеми городами.

In [ ]:
function create_migration_matrix(C, intensity)
    M = ones(C, C) .* intensity ./ (C-1)
    for i = 1:C
        M[i, i] = 1 - intensity
    end
    return M
end

## 3. Функция измерения времени достижения пика

Для заданных параметров запускает симуляцию и возвращает:
- `peak_time` — день, когда доля инфицированных максимальна
- `peak_value` — максимальная доля инфицированных

In [ ]:
function peak_time(p)
    migration_rates = create_migration_matrix(p[:C], p[:migration_intensity])

    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = p[:β_und],
        β_det = p[:β_det],
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
        migration_rates = migration_rates,
    )

    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    peak = 0.0
    peak_step = 0

    for step = 1:p[:n_steps]
        Agents.step!(model, 1)
        frac = infected_frac(model)
        if frac > peak
            peak = frac
            peak_step = step
        end
    end

    return (peak_time = peak_step, peak_value = peak)
end

## 4. Параметры сканирования

- Интенсивность миграции: от 0.0 до 0.5 с шагом 0.1
- Для каждого значения выполняется 3 прогона с разными seed (42, 43, 44)
- Остальные параметры фиксированы
- Инфекция начинается только в первом городе: `Is = [1, 0, 0]`

In [ ]:
migration_intensities = 0.0:0.1:0.5
seeds = [42, 43, 44]

Создаём список параметров

In [ ]:
params_list = []
for mig in migration_intensities
    for s in seeds
        push!(
            params_list,
            Dict(
                :migration_intensity => mig,
                :C => 3,
                :Ns => [1000, 1000, 1000],
                :β_und => [0.5, 0.5, 0.5],
                :β_det => [0.05, 0.05, 0.05],
                :infection_period => 14,
                :detection_time => 7,
                :death_rate => 0.02,
                :reinfection_probability => 0.1,
                :Is => [1, 0, 0],
                :seed => s,
                :n_steps => 150,
            )
        )
    end
end

## 5. Запуск экспериментов

Для каждой комбинации параметров запускается симуляция,
результаты собираются в список.

In [ ]:
results = []
for params in params_list
    data = peak_time(params)
    push!(results, merge(params, Dict(pairs(data))))
    println("Завершен эксперимент с migration_intensity = $(params[:migration_intensity]), seed = $(params[:seed])")
end

## 6. Сохранение результатов

Все прогоны сохраняются в CSV-файл для последующего анализа.

In [ ]:
df = DataFrame(results)
CSV.write(datadir("migration_scan_all.csv"), df)

## 7. Усреднение по повторным прогонам

Для каждого значения интенсивности миграции усредняем показатели по трём seed.

In [ ]:
grouped = combine(
    groupby(df, [:migration_intensity]),
    :peak_time => mean => :mean_peak_time,
    :peak_value => mean => :mean_peak_value,
)

## 8. Визуализация результатов

Строим график зависимости от интенсивности миграции:
- Время достижения пика (в днях)
- Пиковая численность инфицированных (умноженная на 3000 для перевода в абсолютные числа)

In [ ]:
plot(
    grouped.migration_intensity,
    grouped.mean_peak_time,
    marker = :circle,
    xlabel = "Интенсивность миграции",
    ylabel = "Время до пика (дни)",
    label = "Время пика",
)
plot!(
    grouped.migration_intensity,
    grouped.mean_peak_value .* 3000,
    marker = :square,
    xlabel = "Интенсивность миграции",
    ylabel = "Численность в пике",
    label = "Пиковая заболеваемость",
)
savefig(plotsdir("migration_effect.png"))

## 9. Вывод информации о завершении

Ожидаемый результат: с ростом интенсивности миграции пик наступает раньше,
но его высота увеличивается, так как инфекция быстрее распространяется между городами.

In [ ]:
println("Результаты сохранены в data/migration_scan_all.csv и plots/migration_effect.png")